# LIMA employment modelling
Run the cells from top to bottom (**Runtime > Run all** also works). Steps 2-5 each add their sectors to **raw_data.xlsx**; Steps 6-7 draw the report charts.

## Step 1 - Setup
Run this cell first. It asks you to upload **employment_modelling_input.xlsx**. Optionally select at the same time your previous **raw_data.xlsx** (keeps sheets the cells below do not recreate, e.g. *BuildingEnergyEfficiency*) and **ems.xlsx** (Building Energy Efficiency figures).

To work in Google Drive instead, set `USE_GOOGLE_DRIVE = True` and put the input workbook in `MyDrive/LIMA_Employment_modelling/Input_data/`.

In [ ]:
# =============================================================================
# STEP 1 - SETUP (run this cell first)
# Settings, file upload and the helper functions used by all the other cells.
# =============================================================================
import math
import os
import re
import shutil
from collections import namedtuple

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.ticker import FuncFormatter
from openpyxl import Workbook, load_workbook
from openpyxl.utils import range_boundaries

try:
    from google.colab import drive, files  # Only available inside Google Colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# -----------------------------------------------------------------------------
# Settings
# -----------------------------------------------------------------------------
# False: you upload the input workbook now, and download all results at the end.
# True:  files are read from / written to your Google Drive folder WORK_DIR.
USE_GOOGLE_DRIVE = False

if IN_COLAB and USE_GOOGLE_DRIVE:
    drive.mount('/content/drive')
    WORK_DIR = '/content/drive/MyDrive/LIMA_Employment_modelling'
elif IN_COLAB:
    WORK_DIR = '/content/LIMA_Employment_modelling'
else:
    WORK_DIR = os.path.abspath('LIMA_Employment_modelling')  # When run outside Colab

INPUT_FILE = os.path.join(WORK_DIR, 'Input_data', 'employment_modelling_input.xlsx')
OUTPUT_DIR = os.path.join(WORK_DIR, 'Output_data')
RAW_DATA_FILE = os.path.join(OUTPUT_DIR, 'raw_data.xlsx')  # Every model cell adds its sheet(s) here
EMS_FILE = os.path.join(OUTPUT_DIR, 'ems.xlsx')            # Optional: Building Energy Efficiency (report figures)
REPORT_DIR = os.path.join(OUTPUT_DIR, 'Employment_Line_Charts')  # Final report figures
os.makedirs(os.path.dirname(INPUT_FILE), exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# -----------------------------------------------------------------------------
# Upload the input workbook (and, optionally, raw_data.xlsx and ems.xlsx)
# -----------------------------------------------------------------------------
if IN_COLAB and not USE_GOOGLE_DRIVE:
    print('Select employment_modelling_input.xlsx.\n'
          'Optional, in the same window:\n'
          '  - your previous raw_data.xlsx: sheets no cell below recreates (e.g. BuildingEnergyEfficiency) are kept\n'
          '  - ems.xlsx: for the Building Energy Efficiency report figures')
    for name, content in files.upload().items():
        if 'raw_data' in name.lower():
            target = RAW_DATA_FILE
        elif name.lower().startswith('ems'):
            target = EMS_FILE
        else:
            target = INPUT_FILE
        with open(target, 'wb') as f:
            f.write(content)
        if os.path.exists(name):
            os.remove(name)  # files.upload() also saves a copy in /content
        print(f'Saved {name} -> {target}')

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(f'Input workbook not found: {INPUT_FILE}')

# -----------------------------------------------------------------------------
# Excel helpers (like MATLAB readmatrix / readtable / writecell)
# -----------------------------------------------------------------------------
_workbook_cache = {}


def _open_sheet(path, sheet):
    key = (path, os.path.getmtime(path))
    if key not in _workbook_cache:
        _workbook_cache[key] = load_workbook(path, data_only=True)
    return _workbook_cache[key][sheet]


def read_range(path, sheet, cell_range):
    """Raw cell values of a range such as 'M2:Q25', as a list of rows."""
    ws = _open_sheet(path, sheet)
    min_col, min_row, max_col, max_row = range_boundaries(cell_range)
    return [[ws.cell(r, c).value for c in range(min_col, max_col + 1)]
            for r in range(min_row, max_row + 1)]


def to_number(value):
    if isinstance(value, bool) or not isinstance(value, (int, float)):
        return np.nan  # Empty or text cells become NaN, as in MATLAB
    return float(value)


def read_numbers(path, sheet, cell_range):
    """Like MATLAB readmatrix: a 2-D array of numbers."""
    return np.array([[to_number(v) for v in row] for row in read_range(path, sheet, cell_range)])


def read_column(path, sheet, cell_range):
    return read_numbers(path, sheet, cell_range)[:, 0]


def read_value(path, sheet, cell_range):
    return read_numbers(path, sheet, cell_range)[0, 0]


def make_valid_name(name):
    """Mimic MATLAB's variable naming in readtable ('_s1' -> 'x_s1', 'psi_bar Max' -> 'psi_barMax')."""
    name = str(name).strip()
    name = re.sub(r'\s+(\w)', lambda m: m.group(1).upper(), name)
    name = re.sub(r'\s+', '', name)
    name = re.sub(r'\W', '_', name, flags=re.ASCII)
    if not name or not name[0].isalpha():
        name = 'x' + name
    return name


def read_table(path, sheet, cell_range):
    """Like MATLAB readtable: first row = column names, numeric data below (empty rows skipped)."""
    rows = read_range(path, sheet, cell_range)
    header = [make_valid_name(h) for h in rows[0]]
    data = [[to_number(v) for v in row] for row in rows[1:]
            if any(v is not None and str(v).strip() != '' for v in row)]
    return pd.DataFrame(data, columns=header)


Job = namedtuple('Job', 'name psi start length')


def read_job_table(path, sheet, cell_range):
    """Job-role table without header: Var2 = name, Var3 = psi, Var4 = start (B/M/E), Var5 = length (S/M/L)."""
    jobs = []
    for row in read_range(path, sheet, cell_range):
        if all(v is None or str(v).strip() == '' for v in row):
            continue
        name, psi, start, length = row[1], row[2], row[3], row[4]
        jobs.append(Job('' if name is None else str(name), to_number(psi),
                        '' if start is None else str(start), '' if length is None else str(length)))
    return jobs


def _to_excel(value):
    if isinstance(value, np.generic):
        value = value.item()
    if isinstance(value, float) and np.isnan(value):
        return None
    return value


def write_cells(path, sheet, rows, replace_sheet=False):
    """Like MATLAB writecell with 'Sheet': write rows from cell A1, keeping the other sheets.
    replace_sheet=True clears the sheet first (used for raw_data.xlsx)."""
    os.makedirs(os.path.dirname(path), exist_ok=True)
    if os.path.exists(path):
        wb = load_workbook(path)
    else:
        wb = Workbook()
        wb.remove(wb.active)
    if sheet in wb.sheetnames and replace_sheet:
        index = wb.sheetnames.index(sheet)
        wb.remove(wb[sheet])
        ws = wb.create_sheet(sheet, index)
    elif sheet in wb.sheetnames:
        ws = wb[sheet]
    else:
        ws = wb.create_sheet(sheet)
    for r, row in enumerate(rows, start=1):
        for c, value in enumerate(row, start=1):
            ws.cell(r, c).value = _to_excel(value)
    wb.save(path)


# -----------------------------------------------------------------------------
# raw_data.xlsx helpers
# Layout of every sheet: Time Period | Year | then 5 groups of Max, Med, Min,
# 5 columns apart (C-E, H-J, M-O, R-T, W-Y). Groups 1-4 are the scenarios,
# group 5 is the government plan (where a model has one).
# -----------------------------------------------------------------------------
RAW_SCENARIOS = ['ConstCurrent', 'RagingStorm', 'RisingTide', 'ShiftingWinds']
RAW_COLUMN_OFFSET = 5
RAW_FIRST_YEAR = 2022.5  # Period 1 of the half-year grid used for hydrogen, PV and Wind
RAW_NUM_PERIODS = 36     # 2022.5 ... 2040


def raw_header():
    header = ['Time Period', 'Year'] + [None] * (RAW_COLUMN_OFFSET * 5)
    for s in range(5):
        for j, level in enumerate(['Max', 'Med', 'Min']):
            name = f'{RAW_SCENARIOS[s]} {level}' if s < 4 else f'{level}_Employment_S5'
            header[2 + s * RAW_COLUMN_OFFSET + j] = name
    return header


def raw_rows_from_output_table(output_rows):
    """A model output table (Time Period, Year, Max/Med/Min_Employment_S1...) with the raw_data headers."""
    header = raw_header()
    rows = [header]
    for row in output_rows[1:]:
        rows.append(list(row) + [None] * max(0, len(header) - len(row)))
    return rows


def raw_rows_by_year(years, groups):
    """Put results that have their own years onto the 36 half-year periods (2022.5-2040).

    groups: list of (max, med, min) arrays, one per scenario, aligned with years.
    Periods without results are 0; the Year column shows whole years only.
    """
    position = {round(float(y) * 2): i for i, y in enumerate(years)}
    rows = [raw_header()]
    for p in range(1, RAW_NUM_PERIODS + 1):
        year = RAW_FIRST_YEAR + 0.5 * (p - 1)
        row = [p, int(year) if year.is_integer() else None] + [None] * (RAW_COLUMN_OFFSET * 5)
        i = position.get(round(year * 2))
        for s, group in enumerate(groups):
            for j in range(3):
                row[2 + s * RAW_COLUMN_OFFSET + j] = 0 if i is None else group[j][i]
        rows.append(row)
    return rows


def write_raw_data_sheet(sheet, rows):
    write_cells(RAW_DATA_FILE, sheet, rows, replace_sheet=True)
    print(f'raw_data.xlsx: sheet "{sheet}" updated')


# -----------------------------------------------------------------------------
# Chart helper shared by the steel/aluminium/cement and manufacturing cells
# -----------------------------------------------------------------------------
def plot_scenario_bands(plot_years, y_max, y_med, y_min, plot_year_indices, names, colors, title,
                        integer_ticks=False, xticks=None):
    """Median line, shaded min-max band and dashed min/max lines per scenario (12 x 8 cm figure)."""
    fig, ax = plt.subplots(figsize=(12 / 2.54, 8 / 2.54))
    for s in range(len(y_max)):
        ax.fill_between(plot_years, y_min[s][plot_year_indices], y_max[s][plot_year_indices],
                        color=colors[s], alpha=0.2, edgecolor='none')
        ax.plot(plot_years, y_med[s][plot_year_indices], color=colors[s], linewidth=1.5, label=names[s])
        ax.plot(plot_years, y_max[s][plot_year_indices], '--', color=colors[s], linewidth=1)
        ax.plot(plot_years, y_min[s][plot_year_indices], '--', color=colors[s], linewidth=1)
    ax.set_title(title, fontsize=10, fontweight='normal')
    ax.set_xlabel('Years', fontsize=11)
    ax.set_ylabel('Number of Jobs', fontsize=12)
    if xticks is not None:
        ax.set_xticks(xticks)
    if integer_ticks:
        ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0f}'))
    ax.tick_params(labelsize=10)
    ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.2), ncol=2, frameon=False, fontsize=10)
    ax.grid(True)
    fig.tight_layout()


print('Setup complete.')
print(f'  Input workbook: {INPUT_FILE}')
print(f'  Results folder: {OUTPUT_DIR}')

## Step 2 - Steel, aluminium and cement
Adds the sheets *steel*, *aluminium* and *cement* to raw_data.xlsx.

In [ ]:
# =============================================================================
# STEP 2 - STEEL, ALUMINIUM, CEMENT (from the aluminium/cement/steel MATLAB script)
# Arctan employment factor, Max/Med/Min psi, 4 scenarios + government plan.
# Writes: Output_data/Sectors/<Sector>_scenarios.xlsx
#         raw_data.xlsx sheets: aluminium, cement, steel
# =============================================================================
plt.close('all')

sector_steepness_factor = 10
sector_correction_factor = 2800 / 4393
sector_column_offset = 5  # Offset between scenarios in the output file
SECTORS_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'Sectors')

# (input sheet, scenario range, sector parameter range, raw_data sheet)
BASIC_SECTORS = [
    ('Aluminium', 'A1:G32', 'H1:L2', 'aluminium'),
    ('Cement',    'A1:G36', 'H1:L2', 'cement'),
    ('Steel',     'A1:G37', 'H1:L2', 'steel'),
]

SCENARIO_COLORS_4 = [
    (60 / 255, 106 / 255, 255 / 255),   # Blue for Constant Current
    (214 / 255, 69 / 255, 69 / 255),    # Red for Raging Storm
    (13 / 255, 191 / 255, 96 / 255),    # Green for Rising Tide
    (233 / 255, 162 / 255, 126 / 255),  # Orange for Shifting Winds
]
SCENARIO_NAMES_4 = ['Constant Current', 'Raging Storm', 'Rising Tide', 'Shifting Winds']


def arctan_employment_factor(psi, num_periods):
    """A(row, col) = psi * correction_factor * atan(steepness * (row - col + 1)) on and below the diagonal."""
    A = np.zeros((num_periods, num_periods))
    for row in range(1, num_periods + 1):
        for col in range(1, row + 1):
            t_shifted = row - col + 1
            A[row - 1, col - 1] = psi * sector_correction_factor * np.arctan(sector_steepness_factor * t_shifted)
    return A


def scenario_output_table(all_years, y_max, y_med, y_min, total_columns):
    """Time Period | Year | Max/Med/Min_Employment_S1 | 2 blank | ..._S2 | ... (as the MATLAB export)."""
    n = len(all_years)
    table = [[None] * total_columns for _ in range(n + 1)]
    table[0][0], table[0][1] = 'Time Period', 'Year'
    for r in range(n):
        table[r + 1][0] = r + 1
        table[r + 1][1] = all_years[r]
    for s in range(len(y_max)):
        col_start = 2 + s * sector_column_offset
        for j, (label, data) in enumerate(zip(['Max', 'Med', 'Min'], (y_max[s], y_med[s], y_min[s]))):
            table[0][col_start + j] = f'{label}_Employment_S{s + 1}'
            for r in range(n):
                table[r + 1][col_start + j] = data[r]
    return table


for sheet, scenario_range, parameter_range, raw_sheet in BASIC_SECTORS:
    scenarios_data = read_table(INPUT_FILE, sheet, scenario_range)
    all_years = scenarios_data['Year'].to_numpy()
    scenario_vectors = [scenarios_data[c].to_numpy() for c in ['x_s1', 'x_s2', 'x_s3', 'x_s4', 'x_gov']]

    sector_data = read_table(INPUT_FILE, sheet, parameter_range)
    psi_levels = [sector_data[c].iloc[0] for c in ['psi_barMax', 'psi_barMed', 'psi_barMin']]
    num_periods = len(all_years)

    A_max, A_med, A_min = (arctan_employment_factor(psi, num_periods) for psi in psi_levels)
    y_max = [A_max @ x for x in scenario_vectors]
    y_med = [A_med @ x for x in scenario_vectors]
    y_min = [A_min @ x for x in scenario_vectors]

    # Export (all years), then add the sheet to raw_data.xlsx
    table = scenario_output_table(all_years, y_max, y_med, y_min, 2 + 5 * sector_column_offset)
    write_cells(os.path.join(SECTORS_OUTPUT_DIR, f'{sheet}_scenarios.xlsx'), 'Sheet1', table)
    write_raw_data_sheet(raw_sheet, raw_rows_from_output_table(table))

    # Charts for 2025-2040
    plot_year_indices = np.where((all_years >= 2025) & (all_years <= 2040))[0]
    plot_years = all_years[plot_year_indices]
    plot_scenario_bands(plot_years, y_max[:4], y_med[:4], y_min[:4], plot_year_indices, SCENARIO_NAMES_4,
                        SCENARIO_COLORS_4, f'Scenarios for {sheet} sector', xticks=range(2025, 2041, 5))
    plot_scenario_bands(plot_years, y_max[4:], y_med[4:], y_min[4:], plot_year_indices, ['Government Plan'],
                        [(128 / 255, 128 / 255, 128 / 255)], f'Government Plan for {sheet} sector',
                        integer_ticks=True, xticks=range(2025, 2041, 5))

plt.show()

## Step 3 - Wind turbine, PV and electrolyser manufacturing
Adds the sheets *WindMan*, *PVman* and *ElecMan* to raw_data.xlsx. Run Step 2 first.

In [ ]:
# =============================================================================
# STEP 3 - WIND TURBINE, PV AND ELECTROLYSER MANUFACTURING
# (from PV_Wind_Elec_Manufacturing.m)
# Arctan employment factor, Max/Med/Min psi, 4 scenarios + 100% / 50% government plan.
# Writes: raw_data.xlsx sheets: WindMan, PVman, ElecMan
#         (the MATLAB script's own Excel export is switched off, so there is no other file)
# Uses arctan_employment_factor and scenario_output_table from STEP 2 - run STEP 2 first.
# =============================================================================
plt.close('all')

# (input sheet, scenario range, sector parameter range, raw_data sheet)
# Note: the first two sheet names end with a space, as in the MATLAB script.
MANUFACTURING_SECTORS = [
    ('Wind turbine Manufacturing ', 'A1:H37', 'I1:M2', 'WindMan'),
    ('PV Manufacturing ',           'A1:H36', 'I1:M2', 'PVman'),
    ('Hydrogen Production',         'A1:H38', 'I1:M2', 'ElecMan'),  # Electrolyser manufacturing
]
MANUFACTURING_COLUMN_COUNT = 2 + 6 * sector_column_offset
GOVERNMENT_COLORS = [(128 / 255, 128 / 255, 128 / 255), (192 / 255, 192 / 255, 192 / 255)]  # Gray, light gray

for sheet, scenario_range, parameter_range, raw_sheet in MANUFACTURING_SECTORS:
    scenarios_data = read_table(INPUT_FILE, sheet, scenario_range)
    all_years = scenarios_data['Year'].to_numpy()
    scenario_vectors = [scenarios_data[c].to_numpy() for c in ['x_s1', 'x_s2', 'x_s3', 'x_s4']]
    government_vectors = [scenarios_data[c].to_numpy() for c in ['x100_gov', 'x50_gov']]

    sector_data = read_table(INPUT_FILE, sheet, parameter_range)
    psi_levels = [sector_data[c].iloc[0] for c in ['psi_barMax', 'psi_barMed', 'psi_barMin']]
    num_periods = len(all_years)

    A_max, A_med, A_min = (arctan_employment_factor(psi, num_periods) for psi in psi_levels)
    y_max, y_med, y_min = ([A @ x for x in scenario_vectors] for A in (A_max, A_med, A_min))
    gov_max, gov_med, gov_min = ([A @ x for x in government_vectors] for A in (A_max, A_med, A_min))

    # The 4 scenarios go to raw_data.xlsx (government plans are not exported, as in MATLAB)
    table = scenario_output_table(all_years, y_max, y_med, y_min, MANUFACTURING_COLUMN_COUNT)
    write_raw_data_sheet(raw_sheet, raw_rows_from_output_table(table))

    # Charts for 2025-2040
    plot_year_indices = np.where((all_years >= 2025) & (all_years <= 2040))[0]
    plot_years = all_years[plot_year_indices]
    plot_scenario_bands(plot_years, y_max, y_med, y_min, plot_year_indices, SCENARIO_NAMES_4,
                        SCENARIO_COLORS_4, f'Employment Projections for {sheet}', integer_ticks=True)
    plot_scenario_bands(plot_years, gov_max, gov_med, gov_min, plot_year_indices,
                        ['100% Government Plan', '50% Government Plan'], GOVERNMENT_COLORS,
                        f'Government Employment Projections for {sheet}')

plt.show()

## Step 4 - Hydrogen sector
Adds the sheet *hydrogen* to raw_data.xlsx.

In [ ]:
# =============================================================================
# STEP 4 - HYDROGEN SECTOR (from H2.m)
# Four subsectors (Solar, Wind, Electrolyser, Ammonia) with median/upper/lower psi.
# Writes: Output_data/Hydrogen/H2_Sector_New.xlsx and three .svg charts
#         raw_data.xlsx sheet: hydrogen
# =============================================================================
plt.close('all')

H2_OUTPUT_DIR = os.path.join(OUTPUT_DIR, 'Hydrogen')
H2_OUTPUT_FILE = os.path.join(H2_OUTPUT_DIR, 'H2_Sector_New.xlsx')
os.makedirs(H2_OUTPUT_DIR, exist_ok=True)

h2_sheet = 'Hydrogen As one Sector'
h2_time_range = 'B2:B33'   # Sets the number of periods
h2_start_year = 2024.5     # Each period is half a year
H2_INVESTMENT_RANGES = {
    'x_s1': 'C2:C33', 'x_s2': 'D2:D33', 'x_s3': 'E2:E33', 'x_s4': 'F2:F33',
    'x_gov': 'G2:G33',      # Government investment (smoothed)
    'x_gov_cur': 'H2:H33',  # Current government investment (scattered)
}
H2_SUBSECTORS = ['Solar', 'Wind', 'Electrolyser', 'Ammonia']
# psi cells: median (column U), upper (column V), lower (column T). GOV psi (P2, P3, P5, P6) not used.
H2_PSI_CELLS = {
    'median': {'Solar': 'U6:U6', 'Wind': 'U7:U7', 'Electrolyser': 'U9:U9', 'Ammonia': 'U10:U10'},
    'upper':  {'Solar': 'V6:V6', 'Wind': 'V7:V7', 'Electrolyser': 'V9:V9', 'Ammonia': 'V10:V10'},
    'lower':  {'Solar': 'T6:T6', 'Wind': 'T7:T7', 'Electrolyser': 'T9:T9', 'Ammonia': 'T10:T10'},
}
h2_d_map = {'Solar': 1, 'Wind': 1, 'Electrolyser': 1, 'Ammonia': 1}      # Example values, as in H2.m
h2_theta_map = {'Solar': 1, 'Wind': 1, 'Electrolyser': 1, 'Ammonia': 1}  # Example values, as in H2.m
h2_q, h2_k, h2_l = 0.2, 0.49, 1  # Peak height, power of sine (rise), power of cosine (decline)

H2_AREA_COLORS = [(209 / 255, 86 / 255, 86 / 255), (76 / 255, 175 / 255, 80 / 255),
                  (214 / 255, 160 / 255, 118 / 255), (165 / 255, 105 / 255, 189 / 255)]
H2_SCENARIOS = [  # (investment vector, name, colour, export name)
    ('x_s1', 'Constant Current', (0, 0.4470, 0.7410), 'ConstCurrent'),
    ('x_s2', 'Raging Storm', (0.8500, 0.3250, 0.0980), 'RagingStorm'),
    ('x_s3', 'Rising Tide', (0.9290, 0.6940, 0.1250), 'RisingTide'),
    ('x_s4', 'Shifting Winds', (0.4940, 0.1840, 0.5560), 'ShiftingWinds'),
]


def h2_employment_factor(psi_bar, d, num_periods, q, k, l, theta):
    """Sine rise, peak, cosine decline, then psi_bar after d periods."""
    A = np.zeros((num_periods, num_periods))
    for row in range(1, num_periods + 1):
        for col in range(1, row + 1):
            t_diff = row - col + 1
            if t_diff < d / 3:
                value = theta * psi_bar * abs(np.sin((3 * np.pi / (2 * d)) * t_diff)) ** k
            elif t_diff < 2 * d / 3:
                value = q * abs(np.sin((3 * np.pi / d) * (t_diff - d / 3))) + theta * psi_bar
            elif t_diff < d:
                value = (psi_bar * (theta - 1) / 2) * np.cos((3 * np.pi / d) * (t_diff - 2 * d / 3)) ** l \
                    + psi_bar * (1 + theta) / 2
            else:
                value = psi_bar  # After d steps, it settles at psi_bar
            A[row - 1, col - 1] = value
    return A


def h2_axes(ax, title, label_size):
    ax.set_title(title, fontsize=11, fontweight='bold')
    ax.set_xlabel('Years', fontsize=label_size)
    ax.set_ylabel('Number of Jobs', fontsize=label_size)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0f}'))
    ax.grid(True)


# --- Read data ---
h2_x = {name: read_column(INPUT_FILE, h2_sheet, rng) for name, rng in H2_INVESTMENT_RANGES.items()}
h2_num_periods = len(read_column(INPUT_FILE, h2_sheet, h2_time_range))
h2_psi = {level: {s: read_value(INPUT_FILE, h2_sheet, cell) for s, cell in cells.items()}
          for level, cells in H2_PSI_CELLS.items()}

h2_years = h2_start_year + np.arange(h2_num_periods) * 0.5
h2_full_year_indices = np.where(h2_years >= 2025)[0]  # Only show data from 2025 onward

h2_A = {level: {s: h2_employment_factor(h2_psi[level][s], h2_d_map[s], h2_num_periods,
                                        h2_q, h2_k, h2_l, h2_theta_map[s])
                for s in H2_SUBSECTORS}
        for level in H2_PSI_CELLS}


def h2_jobs_by_subsector(level, investment):
    return np.column_stack([h2_A[level][s] @ investment for s in H2_SUBSECTORS])


# --- Stacked area charts for the two government plans (median psi) ---
h2_gov_data = h2_jobs_by_subsector('median', h2_x['x_gov'])[h2_full_year_indices, :]
h2_gov_cur_data = h2_jobs_by_subsector('median', h2_x['x_gov_cur'])[h2_full_year_indices, :]
for data, title, svg_name in [(h2_gov_data, 'Hydrogen sector (smoothed inv.)', 'Employment_Distribution_x_gov.svg'),
                              (h2_gov_cur_data, 'Hydrogen sector (scattered inv.)',
                               'Employment_Distribution_x_gov_cur.svg')]:
    fig, ax = plt.subplots(figsize=(5.6, 4.2))
    areas = ax.stackplot(h2_years[h2_full_year_indices], data.T, colors=H2_AREA_COLORS, linewidth=0)
    ax.set_xlim(h2_years[h2_full_year_indices][0], h2_years[-1])
    h2_axes(ax, title, 10)
    ax.legend(areas, H2_SUBSECTORS, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2,
              frameon=False, fontsize=10)
    fig.tight_layout()
    fig.savefig(os.path.join(H2_OUTPUT_DIR, svg_name))

# --- Total jobs per scenario with upper and lower bounds ---
h2_totals = {name: {level: h2_jobs_by_subsector(level, h2_x[name]).sum(axis=1)
                    for level in ('median', 'upper', 'lower')}
             for name, _, _, _ in H2_SCENARIOS}

fig, ax = plt.subplots(figsize=(5.6, 4.2))
handles = []
for name, label, color, _ in H2_SCENARIOS:
    t = h2_totals[name]
    ax.fill_between(h2_years, t['lower'], t['upper'], color=color, alpha=0.1, linewidth=0)
    line, = ax.plot(h2_years, t['median'], linewidth=2, color=color, label=label)
    ax.plot(h2_years, t['upper'], '--', linewidth=1, color=color)
    ax.plot(h2_years, t['lower'], '--', linewidth=1, color=color)
    handles.append(line)
h2_axes(ax, 'Hydrogen sector (scenarios)', 12)
ax.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, -0.15), ncol=2, frameon=False, fontsize=10)
ax.set_xlim(2025, 2040)
fig.tight_layout()
fig.savefig(os.path.join(H2_OUTPUT_DIR, 'Scenario_Line_Plot_with_Bounds.svg'))

# --- Export the chart data to H2_Sector_New.xlsx ---
for sheet_name, data in [('Raw_x_gov', h2_gov_data), ('Raw_x_gov_cur', h2_gov_cur_data)]:
    rows = [[year] + list(values) for year, values in zip(h2_years[h2_full_year_indices], data)]
    write_cells(H2_OUTPUT_FILE, sheet_name, [['Year'] + H2_SUBSECTORS] + rows)

header = ['Year']
for _, _, _, export_name in H2_SCENARIOS:
    header += [f'{export_name} Max', f'{export_name} Med', f'{export_name} Min']
rows = [[h2_years[p]] + [v for name, _, _, _ in H2_SCENARIOS
                         for v in (h2_totals[name]['upper'][p], h2_totals[name]['median'][p],
                                   h2_totals[name]['lower'][p])]
        for p in range(h2_num_periods)]
write_cells(H2_OUTPUT_FILE, 'Raw_Scenarios', [header] + rows)

# --- raw_data.xlsx: the same scenario data, on the 2022.5-2040 half-year grid ---
write_raw_data_sheet('hydrogen', raw_rows_by_year(
    h2_years, [(h2_totals[name]['upper'], h2_totals[name]['median'], h2_totals[name]['lower'])
               for name, _, _, _ in H2_SCENARIOS]))

plt.show()

## Step 5 - Solar PV and wind power
Job roles (planning, construction, O&M) and phase totals. Adds the sheets *PV* and *Wind* to raw_data.xlsx.

In [ ]:
# =============================================================================
# STEP 5 - SOLAR PV AND WIND POWER
# (Planning.m, Construction.m, OperationAndMaintenance.m, Solar_plots.m, Wind_plots.m)
# Part A: employment by job role -> PV_Wind_Power/employment_per_jobroles/Output_data
# Part B: employment by phase + charts -> PV_Wind_Power/step1 and PV_Wind_Power/scenarios_data
# Writes raw_data.xlsx sheets: PV (Solar), Wind
# =============================================================================
plt.close('all')

PV_WIND_DIR = os.path.join(OUTPUT_DIR, 'PV_Wind_Power')
JOBROLES_DIR = os.path.join(PV_WIND_DIR, 'employment_per_jobroles', 'Output_data')  # Part A
STEP1_DIR = os.path.join(PV_WIND_DIR, 'step1')                                      # Part B, steps B1-B2
SCENARIOS_DATA_DIR = os.path.join(PV_WIND_DIR, 'scenarios_data')                    # Part B, steps B3-B4


# =============================================================================
# Model inputs
# =============================================================================

# --- Planning & construction curve: sine rise -> flat peak -> cosine decline ---
q = 0.0        # The height of the curve's peak
k = 0.49       # Power of sine in the first part (rate of increase)
l = 1          # Power of cosine in the third part (rate of decrease)
theta = 1.009  # Scaling parameter

# --- Operation & maintenance curve: arctan (rises quickly, then stays flat) ---
steepness_factor = 10
correction_factor = 2800 / 4393

DURATION_CELL = 'I2:I2'  # Project duration d, on the Solar and Wind sheets

# --- Part A: employment by job role -----------------------------------------
# (scenario name, input column); rows 2-38
JOBROLE_SCENARIOS = [
    ('x_s1', 'C'), ('x_s2', 'D'), ('x_s3', 'E'), ('x_s4', 'F'),
    ('x_gov', 'G'),
    ('x_gov_new', 'G'),  # As in the MATLAB scripts, this reads column G again (same as x_gov)
]
JOBROLE_ROWS = (2, 38)
JOBROLE_YEARS_RANGE = 'A2:A38'
JOBROLE_OM_OFFSET = -0.0011  # Only OperationAndMaintenance.m subtracts this

# (output sheet name, curve type, job-role table range per technology)
JOBROLE_PHASES = [
    ('Planning',                  'plc', {'Solar': 'M2:Q25',  'Wind': 'M2:Q13'}),   # Planning.m
    ('Construction',              'plc', {'Solar': 'M26:Q64', 'Wind': 'M14:Q38'}),  # Construction.m
    ('Operation and Maintenance', 'om',  {'Solar': 'M65:Q77', 'Wind': 'M39:Q54'}),  # OperationAndMaintenance.m
]

# --- Part B: employment by phase + charts -----------------------------------
# (scenario name, input column); rows 2 to the technology's last_row
PHASE_SCENARIOS = [
    ('x_s1', 'C'), ('x_s2', 'D'), ('x_s3', 'E'), ('x_s4', 'F'),
    ('x_gov_sm', 'G'),  # Government plan, smoothed
    ('x_gov_sc', 'H'),  # Government plan, scattered
]
PLOT_SCENARIOS = ['x_s1', 'x_s2', 'x_s3', 'x_s4', 'x_gov_sc', 'x_gov_sm']
LEGEND_NAMES = ['Constant Current', 'Raging Storm', 'Rising Tide', 'Shifting Winds', 'Scattered', 'Smoothed']
STAGE_NAMES = ['Project Planning and Development', 'Construction', 'Operation and Maintenance']
STAGE_COLORS = [(74 / 255, 111 / 255, 165 / 255), (209 / 255, 86 / 255, 86 / 255), (214 / 255, 160 / 255, 118 / 255)]


def _rgb(values):
    return [tuple(v / 255 for v in rgb) for rgb in values]


# Where Solar_plots.m and Wind_plots.m differ
TECHNOLOGIES = {
    'Solar': dict(                       # Solar_plots.m
        raw_sheet='PV',                  # Sheet in raw_data.xlsx
        last_row=37,                     # Investment in C2:C37 ... H2:H37 (36 half-year periods)
        plc_job_range='T2:X3',           # GOV ('T9:X10' for the scenarios)
        om_job_range='T4:X4',            # GOV ('T11:X11' for the scenarios)
        # Solar_plots.m names the planning & construction sheets with hyphens
        plc_sheet_names=['x-s1', 'x-s2', 'x-s3', 'x-s4', 'x-gov-sm', 'x-gov-sc'],
        om_range_for_totals='B7:C37',    # O&M rows used for the scenario charts (B3)
        stretch_to_even_years=True,      # B3: 2025:0.5:2040 made 32 long; data stretched by interpolation
        scenario_colors=_rgb([(85, 168, 104), (244, 123, 32), (72, 133, 237),
                              (219, 68, 55), (171, 71, 188), (66, 133, 244)]),
        title_main='Employment Over Time for Scenarios x-s1 to x-s4',
        title_gov='Employment Over Time for Scenarios x-gov-sc and x-gov-sm',
    ),
    'Wind': dict(                        # Wind_plots.m
        raw_sheet='Wind',                # Sheet in raw_data.xlsx
        last_row=32,                     # Investment in C2:C32 ... H2:H32 (31 half-year periods)
        plc_job_range='M55:Q56',         # gov ('T2:X3' for the scenarios)
        om_job_range='M57:Q57',          # Gov ('T3:X3' for the scenarios)
        plc_sheet_names=['x_s1', 'x_s2', 'x_s3', 'x_s4', 'x_gov_sm', 'x_gov_sc'],
        om_range_for_totals='B2:C32',
        stretch_to_even_years=False,
        scenario_colors=_rgb([(85, 168, 104), (244, 123, 32), (72, 133, 237),
                              (219, 68, 55), (85, 168, 104), (66, 133, 244)]),
        title_main='Employment Over Time for Scenarios x_s1 to x_s4',
        title_gov='Employment Over Time for Scenarios x_gov_sc and x_gov_sm',
    ),
}
PLC_RANGE_FOR_CHARTS = 'B7:C37'  # Planning (column B) and construction (column C) rows used in B3 and B4
OM_RANGE_FOR_STACKED = 'B2:B32'  # O&M rows used in B4


# =============================================================================
# Model formulas
# =============================================================================
def convert_start_length(start, length, d, early_share):
    """Turn a job's start (B/M/E) and length (S/M/L) into a time window.

    early_share is 0.2 in Planning.m / Construction.m and 0.25 in Solar_plots.m / Wind_plots.m.
    """
    start_time = end_time = None
    if start == 'B':    # Beginning
        start_time = 0
        end_time = {'S': early_share * d, 'M': 0.8 * d, 'L': d}.get(length)
    elif start == 'M':  # Middle
        start_time = early_share * d
        end_time = {'S': 0.8 * d, 'M': d}.get(length)
    elif start == 'E':  # End
        start_time = 0.8 * d
        end_time = {'S': d}.get(length)
    if start_time is None or end_time is None:
        # MATLAB stops with "Output argument not assigned" for these combinations
        raise ValueError(f'Unsupported start/length combination: {start!r}/{length!r}')
    return start_time, end_time


def plc_factor_matrix(psi, start_time, end_time, d, num_periods, formula):
    """Employment factor matrix A for planning & construction jobs.

    formula='jobroles' is the version in Planning.m / Construction.m and
    formula='plots' the version in Solar_plots.m / Wind_plots.m. They differ
    only in the peak and decline pieces; with q = 0 and l = 1 both give the same numbers.
    """
    A = np.zeros((num_periods, num_periods))
    for row in range(1, num_periods + 1):
        for col in range(1, row + 1):
            t = row - col + 1  # Periods since the investment was made
            if not (start_time <= t <= end_time):
                continue
            if t <= d / 3:  # Rise
                value = theta * psi * np.sin((3 * np.pi / (2 * d)) * t) ** k
            elif t <= 2 * d / 3:  # Peak
                bump = q * np.sin((3 * np.pi / d) * (t - d / 3))
                if formula == 'jobroles':
                    bump = abs(bump)
                value = bump + theta * psi
            elif t <= d:  # Decline
                cosine = np.cos((3 * np.pi / d) * (t - 2 * d / 3))
                if formula == 'jobroles':
                    value = (psi * (theta - 1) / 2) * cosine ** l + psi * (1 + theta) / 2
                else:
                    value = (psi * (theta - 1) / 2 * cosine) ** l + psi * (1 + theta) / 2
            else:
                continue
            A[row - 1, col - 1] = value
    return A


def om_factor_matrix(psi, start_time, num_periods, offset):
    """Employment factor matrix A for operation & maintenance jobs (arctan curve).

    offset is -0.0011 in OperationAndMaintenance.m and 0 in Solar_plots.m / Wind_plots.m.
    """
    A = np.zeros((num_periods, num_periods))
    for row in range(1, num_periods + 1):
        for col in range(1, row + 1):
            t = row - col + 1 - start_time
            if t >= 0:
                A[row - 1, col - 1] = psi * correction_factor * np.arctan(steepness_factor * t) + offset
    return A


# =============================================================================
# Part A: employment by job role
#    (Planning.m, Construction.m, OperationAndMaintenance.m)
#    Output: <Tech>_JobRoles_Scenarios.xlsx with sheets Planning, Construction,
#    Operation and Maintenance. In each sheet the six scenarios are stacked
#    vertically, one column per job role.
# =============================================================================
def run_jobrole_phase(tech, phase_name, curve, job_range):
    jobs = read_job_table(INPUT_FILE, tech, job_range)
    first_row, last_row = JOBROLE_ROWS
    scenarios = [(name, read_column(INPUT_FILE, tech, f'{col}{first_row}:{col}{last_row}'))
                 for name, col in JOBROLE_SCENARIOS]

    if curve == 'plc':
        d = read_numbers(INPUT_FILE, tech, DURATION_CELL)[0, 0]
        years = list(read_column(INPUT_FILE, tech, JOBROLE_YEARS_RANGE))  # Years from the input file
    else:
        # O&M: half-year periods labelled 2025, blank, 2026, blank, ...
        num_periods = len(scenarios[0][1])
        years = [2025 + i // 2 if i % 2 == 0 else None for i in range(num_periods)]
    num_periods = len(years)

    all_rows = []
    for scenario_name, x in scenarios:
        # Header row, then one row per period: Scenario | Time Period | Year | job 1 | job 2 | ...
        block = [['Scenario', 'Time Period', 'Year'] + [job.name for job in jobs]]
        block += [[scenario_name, p + 1, years[p]] + [None] * len(jobs) for p in range(num_periods)]

        for j, job in enumerate(jobs):
            if curve == 'plc':
                if job.psi == 0:
                    continue  # Left blank
                start_time, end_time = convert_start_length(job.start, job.length, d, early_share=0.2)
                A = plc_factor_matrix(job.psi, start_time, end_time, d, num_periods, formula='jobroles')
            else:
                A = om_factor_matrix(job.psi, 0, num_periods, offset=JOBROLE_OM_OFFSET)
            y = A @ x
            for p in range(num_periods):
                block[p + 1][3 + j] = y[p]

        all_rows += block

    output_file = os.path.join(JOBROLES_DIR, f'{tech}_JobRoles_Scenarios.xlsx')
    write_cells(output_file, phase_name, all_rows)
    print(f'{tech} - {phase_name}: employment vectors for all scenarios exported to Excel.')


# =============================================================================
# Part B: employment by phase + charts (Solar_plots.m, Wind_plots.m)
#    B1  Planning & construction totals -> step1/<Tech>_Employment_Output_PLC.xlsx
#    B2  Operation & maintenance totals -> step1/<Tech>_Employment_Output_OM.xlsx
#    B3  Scenario charts with +/-20% bands -> scenarios_data/<Tech>_Scenarios_Plot_Data.xlsx
#    B4  Stacked area chart per scenario  -> scenarios_data/<Tech>_Stacked_Plot_Data.xlsx
# =============================================================================
def step_b1_planning_construction(tech, cfg, x, plc_file):
    jobs = read_job_table(INPUT_FILE, tech, cfg['plc_job_range'])
    d = read_numbers(INPUT_FILE, tech, DURATION_CELL)[0, 0]
    num_periods = len(x['x_s1'])
    print(f'{tech} PLC - Number of Jobs: {len(jobs)}')
    print(f'{tech} PLC - Number of Periods: {num_periods}')

    Y = {name: np.zeros((num_periods, len(jobs))) for name in x}
    A_matrices = [None] * len(jobs)
    for i, job in enumerate(jobs):
        if job.psi == 0:
            continue  # Skip if no employment factor
        start_time, end_time = convert_start_length(job.start, job.length, d, early_share=0.25)
        A = plc_factor_matrix(job.psi, start_time, end_time, d, num_periods, formula='plots')
        A_matrices[i] = A
        for name in Y:
            Y[name][:, i] = A @ x[name]

    # One sheet per scenario: period numbers in column A, one column per job
    header = [None] + [job.name for job in jobs]
    for (name, _), sheet_name in zip(PHASE_SCENARIOS, cfg['plc_sheet_names']):
        rows = [header] + [[p + 1] + list(Y[name][p]) for p in range(num_periods)]
        write_cells(plc_file, sheet_name, rows)

    # Each job's A matrix on its own sheet (sheet names cleaned, max 31 characters)
    for job, A in zip(jobs, A_matrices):
        if A is not None:
            sheet_name = re.sub(r'[:\\/*?\[\]]', '', job.name)[:31]
            write_cells(plc_file, sheet_name, A.tolist())
    print(f'{tech} PLC employment vectors and factor matrices exported.')


def step_b2_operation_maintenance(tech, cfg, x, om_file):
    jobs = read_job_table(INPUT_FILE, tech, cfg['om_job_range'])
    num_periods = len(x['x_gov_sm'])
    print(f'{tech} OM - Number of Jobs: {len(jobs)}')
    print(f'{tech} OM - Number of Periods: {num_periods}')

    Y = {name: np.zeros((num_periods, len(jobs))) for name in x}
    for i, job in enumerate(jobs):
        A = om_factor_matrix(job.psi, 0, num_periods, offset=0.0)  # For OM, start time is zero
        for name in Y:
            Y[name][:, i] = A @ x[name]

    header = [None] + [job.name for job in jobs]
    for name, _ in PHASE_SCENARIOS:
        rows = [header] + [[p + 1] + list(Y[name][p]) for p in range(num_periods)]
        write_cells(om_file, name, rows)
    print(f'{tech} OM employment vectors exported.')


def _fill_missing_with_zero(values):
    return np.where(np.isnan(values), 0.0, values)  # MATLAB fillmissing(..., 'constant', 0)


def _stretch(values, length):
    """MATLAB interp1(linspace(1, length, numel(values)), values, 1:length, 'linear', 'extrap')."""
    if len(values) == length:
        return values
    return np.interp(np.arange(1, length + 1), np.linspace(1, length, len(values)), values)


def step_b3_scenario_charts(tech, cfg, plc_file, om_file):
    years = np.arange(2025, 2040.25, 0.5)  # 2025:0.5:2040 -> 31 values
    if cfg['stretch_to_even_years'] and len(years) % 2 != 0:
        years = np.append(years, years[-1] + 0.5)

    plc_sheet = dict(zip([name for name, _ in PHASE_SCENARIOS], cfg['plc_sheet_names']))
    total_employment = np.zeros((len(years), len(PLOT_SCENARIOS)))
    for i, scenario in enumerate(PLOT_SCENARIOS):
        data_plc = _fill_missing_with_zero(read_numbers(plc_file, plc_sheet[scenario], PLC_RANGE_FOR_CHARTS))
        data_om = _fill_missing_with_zero(read_numbers(om_file, scenario, cfg['om_range_for_totals']))
        planning, construction, op_maint = data_plc[:, 0], data_plc[:, 1], data_om[:, 0]
        if cfg['stretch_to_even_years']:
            planning, construction, op_maint = (_stretch(v, len(years)) for v in (planning, construction, op_maint))
        total_employment[:, i] = planning + construction + op_maint

    if cfg['stretch_to_even_years'] and np.isnan(total_employment).any():
        raise ValueError('total_employment contains NaN values. Check the input data files or ranges.')

    upper_bounds = total_employment * 1.2
    lower_bounds = total_employment * 0.8

    # raw_data.xlsx: the four scenarios (Max = upper bound, Med = total, Min = lower bound)
    write_raw_data_sheet(cfg['raw_sheet'], raw_rows_by_year(
        years, [(upper_bounds[:, i], total_employment[:, i], lower_bounds[:, i]) for i in range(4)]))

    plot_file = os.path.join(SCENARIOS_DATA_DIR, f'{tech}_Scenarios_Plot_Data.xlsx')
    for indices, title, sheet in [([0, 1, 2, 3], cfg['title_main'], 'Scenarios'),
                                  ([4, 5], cfg['title_gov'], 'GOV_Scenarios')]:
        plot_scenario_bands(years, total_employment, upper_bounds, lower_bounds, indices,
                            cfg['scenario_colors'], title)
        # Export plot data: Year, then Max / Med / Min for each scenario
        header = ['Year']
        for i in indices:
            header += [f'Max_{PLOT_SCENARIOS[i]}', f'Med_{PLOT_SCENARIOS[i]}', f'Min_{PLOT_SCENARIOS[i]}']
        rows = [[years[r]] + [v for i in indices
                              for v in (upper_bounds[r, i], total_employment[r, i], lower_bounds[r, i])]
                for r in range(len(years))]
        write_cells(plot_file, sheet, [header] + rows)


def step_b4_stacked_charts(tech, cfg, plc_file, om_file):
    years = 2025 + np.arange(31) * 0.5  # 2025:0.5:2040
    plc_sheet = dict(zip([name for name, _ in PHASE_SCENARIOS], cfg['plc_sheet_names']))
    stack_file = os.path.join(SCENARIOS_DATA_DIR, f'{tech}_Stacked_Plot_Data.xlsx')

    for scenario in PLOT_SCENARIOS:
        data_plc = read_numbers(plc_file, plc_sheet[scenario], PLC_RANGE_FOR_CHARTS)
        data_om = read_numbers(om_file, scenario, OM_RANGE_FOR_STACKED)
        planning, construction, op_maint = data_plc[:, 0], data_plc[:, 1], data_om[:, 0]
        if not (len(planning) == len(construction) == len(op_maint) == len(years)):
            raise ValueError(f'Data length mismatch in scenario {scenario}. Verify data ranges.')

        plot_stacked(years, [planning, construction, op_maint], f'Employment Over Time for Scenario: {scenario}')

        header = ['Year', 'Project_Planning_Development', 'Construction', 'Operation_Maintenance']
        rows = [[years[r], planning[r], construction[r], op_maint[r]] for r in range(len(years))]
        write_cells(stack_file, scenario, [header] + rows)


def run_technology(tech, cfg):
    plc_file = os.path.join(STEP1_DIR, f'{tech}_Employment_Output_PLC.xlsx')
    om_file = os.path.join(STEP1_DIR, f'{tech}_Employment_Output_OM.xlsx')
    # Investment vectors for each scenario
    x = {name: read_column(INPUT_FILE, tech, f'{col}2:{col}{cfg["last_row"]}') for name, col in PHASE_SCENARIOS}

    step_b1_planning_construction(tech, cfg, x, plc_file)
    step_b2_operation_maintenance(tech, cfg, x, om_file)
    step_b3_scenario_charts(tech, cfg, plc_file, om_file)
    step_b4_stacked_charts(tech, cfg, plc_file, om_file)
    print(f'{tech}: plotting complete. All data exported.')


# --- Charts (MATLAB default figure size 560 x 420 px, axes at [0.13 0.3 0.8 0.6]) ---
def _new_figure():
    fig = plt.figure(figsize=(5.6, 4.2))
    ax = fig.add_axes([0.13, 0.3, 0.8, 0.6])
    return fig, ax


def _finish_axes(fig, ax, title, handles):
    ax.set_xlabel('Year')
    ax.set_ylabel('Number of Jobs')
    ax.set_title(title, fontsize=10, fontweight='bold')
    ax.set_xticks(range(2025, 2041, 5))
    ax.set_xlim(2025, 2040)
    ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:.0f}'))  # No 10^x exponent
    ax.grid(True)
    # Legend below the axes without a box (2 columns so it fits the figure width)
    fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.5, 0.17), ncol=2, frameon=False, fontsize=9)


def plot_scenario_bands(years, total, upper, lower, indices, colors, title):
    fig, ax = _new_figure()
    handles = []
    for i in indices:
        ax.fill_between(years, lower[:, i], upper[:, i], color=colors[i], alpha=0.3, linewidth=0)
        line, = ax.plot(years, total[:, i], color=colors[i], linewidth=1.5, label=LEGEND_NAMES[i])
        ax.plot(years, upper[:, i], '--', color=colors[i], linewidth=1)
        ax.plot(years, lower[:, i], '--', color=colors[i], linewidth=1)
        handles.append(line)
    _finish_axes(fig, ax, title, handles)


def plot_stacked(years, stages, title):
    fig, ax = _new_figure()
    areas = ax.stackplot(years, *stages, colors=STAGE_COLORS, labels=STAGE_NAMES, linewidth=0)
    _finish_axes(fig, ax, title, areas)


# =============================================================================
# Run Part A and Part B
# =============================================================================
# Part A: employment by job role (Planning, Construction, O&M) for Solar and Wind
for phase_name, curve, job_ranges in JOBROLE_PHASES:
    for tech in ('Solar', 'Wind'):
        run_jobrole_phase(tech, phase_name, curve, job_ranges[tech])

# Part B: employment by phase and charts, Solar then Wind
for tech, cfg in TECHNOLOGIES.items():
    run_technology(tech, cfg)
    plt.show()

## Step 6 - Scenario charts for all sectors
Reads raw_data.xlsx and saves one .svg chart per sector (lineChart.py).

In [ ]:
# =============================================================================
# STEP 6 - SCENARIO CHARTS FOR ALL SECTORS (reads raw_data.xlsx)
# One small chart per sector: median line, min/max band, dashed min/max lines.
# Writes: Output_data/Employment_Line_Charts/<sheet>_scenarios.svg (lineChart.py)
# =============================================================================
plt.close('all')

SVG_DIR = REPORT_DIR

# Sector name -> sheet name in raw_data.xlsx
sector_sheets = {
    'Clean Steel': 'steel',
    'Hydrogen Production and Infrastructure': 'hydrogen',
    'Clean Aluminium': 'aluminium',
    'Clean Cement': 'cement',
    'Building Energy Efficiency': 'BuildingEnergyEfficiency',
    'Wind Turbine Manufacturing': 'WindMan',
    'Electrolyser Manufacturing': 'ElecMan',
    'Solar PV Manufacturing': 'PVman',
    'Solar PV Development': 'PV',
    'Wind Power Development': 'Wind',
}

# Scenario -> (median column, min column, max column, colour)
chart_scenarios = {
    'Constant Current': ('ConstCurrent Med', 'ConstCurrent Min', 'ConstCurrent Max', '#152570'),
    'Raging Storm': ('RagingStorm Med', 'RagingStorm Min', 'RagingStorm Max', '#881620'),
    'Rising Tide': ('RisingTide Med', 'RisingTide Min', 'RisingTide Max', '#074425'),
    'Shifting Winds': ('ShiftingWinds Med', 'ShiftingWinds Min', 'ShiftingWinds Max', '#b37d54'),
}

# Fixed y-axis for some sheets (the others are scaled automatically)
custom_yaxis_settings = {
    'steel': {'new_max': 21000, 'interval': 5000},
    'WindMan': {'new_max': 4200, 'interval': 1000},
    'Wind': {'new_max': 8000, 'interval': 2000},
}

# Figure and styling
chart_figsize = (3.2, 1.5)
label_fontsize = 4
tick_fontsize = 4
spine_linewidth = 1


def y_formatter(x, pos):
    """Y-axis tick labels with thousands separators, e.g. 21,000."""
    return f"{int(x):,}"


def load_sector_data(sheet):
    """Read one sector sheet and keep the rows for 2025-2040."""
    df = pd.read_excel(RAW_DATA_FILE, sheet_name=sheet, engine='openpyxl')

    # Ensure the 'Year' column is numeric and filter rows for 2025-2040
    df = df[pd.to_numeric(df['Year'], errors='coerce').notna()]
    df['Year'] = df['Year'].astype(int)
    df = df[(df['Year'] >= 2025) & (df['Year'] <= 2040)]

    # Convert employment scenario columns to numeric
    emp_cols = [col for col in df.columns if any(keyword in str(col) for keyword in
                ['ConstCurrent', 'RagingStorm', 'RisingTide', 'ShiftingWinds'])]
    df[emp_cols] = df[emp_cols].apply(pd.to_numeric, errors='coerce')
    return df


def y_axis_ticks(df, sheet):
    """Return (new_max, yticks): 4 equal steps from 0, rounded up to a round number."""
    if sheet in custom_yaxis_settings:
        new_max = custom_yaxis_settings[sheet]['new_max']
        interval = custom_yaxis_settings[sheet]['interval']
        return new_max, [interval * i for i in range(5)]

    y_max_value = max(df['ConstCurrent Max'].max(), df['RagingStorm Max'].max(),
                      df['RisingTide Max'].max(), df['ShiftingWinds Max'].max())
    if y_max_value <= 30000:
        factor = 1000
    elif y_max_value <= 100000:
        factor = 5000
    else:
        factor = 10000

    y_ceil = math.ceil(y_max_value / factor) * factor
    raw_interval = y_ceil / 4
    interval = math.ceil(raw_interval / factor) * factor
    new_max = interval * 4
    return new_max, [interval * i for i in range(5)]


def plot_sector(df, sheet):
    """Draw the scenario chart for one sector and save it as an .svg file."""
    years = df['Year']

    plt.figure(figsize=chart_figsize)
    ax = plt.gca()

    for label, (med_col, min_col, max_col, color) in chart_scenarios.items():
        ax.fill_between(years, df[min_col], df[max_col], color=color, alpha=0.3)
        ax.plot(years, df[min_col], linestyle='--', color=color, alpha=0.4, linewidth=0.6)
        ax.plot(years, df[max_col], linestyle='--', color=color, alpha=0.4, linewidth=0.6)
        ax.plot(years, df[med_col], color=color, linewidth=0.6, label=label)

    # Legend (switched off, as in the original)
    # ax.legend(fontsize=tick_fontsize, loc='best')

    ax.set_xlim(2025, 2040)
    plt.xticks(range(2025, 2041, 3), fontsize=tick_fontsize, alpha=1)

    new_max, yticks = y_axis_ticks(df, sheet)
    plt.ylim(0, new_max)
    plt.yticks(yticks, fontsize=tick_fontsize, alpha=1)
    ax.yaxis.set_major_formatter(FuncFormatter(y_formatter))

    for spine in ax.spines.values():
        spine.set_linewidth(spine_linewidth)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_alpha(0.4)
    ax.spines['left'].set_alpha(0.4)

    plt.xlabel('Year', fontsize=label_fontsize, labelpad=1)
    plt.ylabel('Number of Jobs', fontsize=label_fontsize, labelpad=1)

    fig = plt.gcf()
    fig.set_size_inches(*chart_figsize)
    plt.tight_layout()
    plt.savefig(os.path.join(SVG_DIR, f"{sheet}_scenarios.svg"), format="svg", transparent=True)
    plt.show()
    plt.close()


os.makedirs(SVG_DIR, exist_ok=True)
available_sheets = load_workbook(RAW_DATA_FILE, read_only=True).sheetnames
for sector_name, sheet in sector_sheets.items():
    if sheet not in available_sheets:
        print(f'Skipped {sector_name}: raw_data.xlsx has no sheet "{sheet}"')
        continue
    plot_sector(load_sector_data(sheet), sheet)

print("Plots generated and saved for all sectors.")

## Step 7 - Final report figures
Stacked charts for solar, wind and hydrogen, the large steel chart and, if ems.xlsx was uploaded, the Building Energy Efficiency charts. Set `STEEL_COLOURS` and the y-axis settings at the top.

In [ ]:
# =============================================================================
# STEP 7 - FINAL REPORT FIGURES (from the "Python code for EF-plots" scripts)
# Uses the files written by Steps 2, 4 and 5, and ems.xlsx if it was uploaded.
# Writes to Output_data/Employment_Line_Charts:
#   Solar_Stacked_Smoothed.svg, Solar_Stacked_Scattered.svg        (solar_stacked.py)
#   Wind_Stacked_Smoothed_x_gov_sm.svg, ..._x_gov_sc.svg           (wind_stacked.py)
#   Hydrogen_Stacked_Smooth.svg, Hydrogen_Stacked_Scattered.svg    (Stacked_plots.py)
#   plots/steel_scenarios.svg                                      (Steel_*colours_scenarios.py)
#   ems.svg, ems_area_chart.svg                                    (BEE.py, stacked_ems.py)
# =============================================================================
from scipy.interpolate import make_interp_spline

plt.close('all')
os.makedirs(REPORT_DIR, exist_ok=True)

STEEL_COLOURS = 'bright'  # 'bright' (Steel_brightcolours_scenarios.py) or 'dark' (Steel_darkcolours_scenarios.py)

# Per-figure y-axis settings: {"new_max": top of the axis, "interval": tick spacing}
REPORT_YAXIS = {
    'Solar_Stacked_Smoothed': {"new_max": 16000, "interval": 4000},
    'Solar_Stacked_Scattered': {"new_max": 28000, "interval": 7000},
    'Wind_Stacked_Smoothed_x_gov_sm': {"new_max": 3000, "interval": 1000},
    'Wind_Stacked_Smoothed_x_gov_sc': {"new_max": 0, "interval": 1000},
    'Hydrogen Stacked (Smooth)': {"new_max": 30000, "interval": 8000},
    'Hydrogen Stacked (Scattered)': {"new_max": 3000, "interval": 8000},
    'ems': {"new_max": 16000, "interval": 4000},
}


def hundred_formatter(x, pos):
    """Tick labels floored to the nearest hundred, with thousands separators."""
    return f'{(int(x) // 100) * 100:,}'


def plot_report_stackplot(years, series, labels, colors, title, smooth, num_points, custom_yaxis):
    """Stacked area chart in the report format (3.2 x 1.5 in), saved as <title>.svg.

    Same as plot_stackplot in solar_stacked.py / wind_stacked.py / Stacked_plots.py.
    """
    series = [np.asarray(s) for s in series]
    if smooth:
        # Remove duplicate x-values, then cubic-spline onto a finer x-axis
        unique_years, unique_indices = np.unique(years, return_index=True)
        years = unique_years
        series = [s[unique_indices] for s in series]
        xnew = np.linspace(np.min(years), np.max(years), num_points)
        series = [make_interp_spline(years, s, k=3)(xnew) for s in series]
        years = xnew

    # Total jobs, rounded up to the next hundred (automatic y-axis)
    y_max_tick = np.ceil(np.max(np.sum(series, axis=0)) / 100) * 100

    plt.figure(figsize=(3.2, 1.5))
    ax = plt.gca()
    label_fontsize = 4
    tick_fontsize = 4

    ax.stackplot(years, *series, colors=colors, labels=labels)
    # ax.legend(loc='upper left', fontsize=label_fontsize)
    plt.xlabel('Year', fontsize=label_fontsize, labelpad=1)
    plt.ylabel('Number of Jobs', fontsize=label_fontsize, labelpad=1)

    ax.set_xlim(2025, 2040)
    plt.xticks(np.arange(2025, 2041, 5), fontsize=tick_fontsize)

    if custom_yaxis is not None and custom_yaxis.get("new_max") is not None \
            and custom_yaxis.get("interval") is not None:
        new_max = custom_yaxis["new_max"]
        yticks = [custom_yaxis["interval"] * i for i in range(5)]
    else:
        new_max = y_max_tick
        yticks = np.linspace(0, new_max, 5)
    ax.set_ylim(0, new_max)
    plt.yticks(yticks, fontsize=tick_fontsize)
    ax.yaxis.set_major_formatter(FuncFormatter(hundred_formatter))

    for spine in ax.spines.values():
        spine.set_linewidth(1.5)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.spines['bottom'].set_alpha(0.4)
    ax.spines['left'].set_alpha(0.4)

    plt.tight_layout()
    safe_title = title.replace(" ", "_").replace("(", "").replace(")", "").replace("-", "")
    plt.savefig(os.path.join(REPORT_DIR, f"{safe_title}.svg"), format='svg', bbox_inches='tight', transparent=True)
    plt.show()


def read_stacked_sheet(path, sheet, columns, group_by_year):
    """Read a stacked-chart sheet; years become whole numbers (half-years share a year)."""
    df = pd.read_excel(path, sheet_name=sheet)
    df = df.dropna(subset=['Year'])
    if group_by_year:
        df = df.groupby('Year', as_index=False).mean()
    df['Year'] = df['Year'].astype(int)
    # Empty cells count as 0 jobs. (On a fresh run the Wind planning & construction data has no
    # values after mid-2037 - see Step 5 - and the smoothing cannot handle empty cells.)
    return df['Year'].to_numpy(), [df[c].fillna(0).to_numpy() for c in columns]


# -----------------------------------------------------------------------------
# 1. Solar and wind: jobs by phase, government plans (solar_stacked.py, wind_stacked.py)
# -----------------------------------------------------------------------------
PHASE_COLUMNS = ['Project_Planning_Development', 'Construction', 'Operation_Maintenance']
PHASE_LABELS = ['Planning and development', 'Construction and installation', 'Operation and maintenance']
PHASE_COLORS = ['#008080', '#7495AB', '#E6E0D4']

for tech, sm_title, sc_title in [('Solar', 'Solar_Stacked_Smoothed', 'Solar_Stacked_Scattered'),
                                 ('Wind', 'Wind_Stacked_Smoothed_x_gov_sm', 'Wind_Stacked_Smoothed_x_gov_sc')]:
    stacked_file = os.path.join(SCENARIOS_DATA_DIR, f'{tech}_Stacked_Plot_Data.xlsx')
    for sheet, title, group_by_year in [('x_gov_sm', sm_title, True), ('x_gov_sc', sc_title, False)]:
        years, series = read_stacked_sheet(stacked_file, sheet, PHASE_COLUMNS, group_by_year)
        plot_report_stackplot(years, series, PHASE_LABELS, PHASE_COLORS, title,
                              smooth=True, num_points=35, custom_yaxis=REPORT_YAXIS[title])

# -----------------------------------------------------------------------------
# 2. Hydrogen: jobs by subsector, government plans (Stacked_plots.py)
# -----------------------------------------------------------------------------
H2_COLUMNS = ['Solar', 'Wind', 'Electrolyser', 'Ammonia']
H2_REPORT_COLORS = ['#E6E0D4', '#7C1B19', '#36454F', '#7C5B84']  # From bottom to top

for sheet, title, smooth, group_by_year in [('Raw_x_gov', 'Hydrogen Stacked (Smooth)', True, True),
                                            ('Raw_x_gov_cur', 'Hydrogen Stacked (Scattered)', False, False)]:
    years, series = read_stacked_sheet(H2_OUTPUT_FILE, sheet, H2_COLUMNS, group_by_year)
    plot_report_stackplot(years, series, H2_COLUMNS, H2_REPORT_COLORS, title,
                          smooth=smooth, num_points=300, custom_yaxis=REPORT_YAXIS[title])

# -----------------------------------------------------------------------------
# 3. Steel: four scenarios, large figure (Steel_brightcolours / Steel_darkcolours_scenarios.py)
# -----------------------------------------------------------------------------
df = pd.read_excel(os.path.join(SECTORS_OUTPUT_DIR, 'Steel_scenarios.xlsx'), skiprows=6, header=None,
                   engine='openpyxl')
df.columns = ['Time Period', 'Year',
              'Max_Employment_S1', 'Med_Employment_S1', 'Min_Employment_S1', 'Empty1', 'Empty2',
              'Max_Employment_S2', 'Med_Employment_S2', 'Min_Employment_S2', 'Empty3', 'Empty4',
              'Max_Employment_S3', 'Med_Employment_S3', 'Min_Employment_S3', 'Empty5', 'Empty6',
              'Max_Employment_S4', 'Med_Employment_S4', 'Min_Employment_S4', 'Empty7', 'Empty8',
              'Check1', 'Check2', 'Check3']
df = df[pd.to_numeric(df['Year'], errors='coerce').notna()]
df['Year'] = df['Year'].astype(int)
df = df[(df['Year'] >= 2025) & (df['Year'] <= 2040)]
cols_to_convert = [col for col in df.columns if 'Employment' in col]
df[cols_to_convert] = df[cols_to_convert].apply(pd.to_numeric, errors='coerce')
df = df.dropna(subset=cols_to_convert)
years = df['Year']

if STEEL_COLOURS == 'bright':
    steel_colors = ['#2C4FE8', '#BA3E43', '#0FA349', '#DD8956']
else:
    steel_colors = ['#152570', '#881620', '#074425', '#b37d54']

plt.figure(figsize=(12, 7))
ax = plt.gca()
for s, color in enumerate(steel_colors, start=1):
    med, min_, max_ = f'Med_Employment_S{s}', f'Min_Employment_S{s}', f'Max_Employment_S{s}'
    ax.fill_between(years, df[min_], df[max_], color=color, alpha=0.3)
    ax.plot(years, df[min_], linestyle='--', color=color, alpha=0.4, linewidth=1.1)
    ax.plot(years, df[max_], linestyle='--', color=color, alpha=0.4, linewidth=1.1)
    ax.plot(years, df[med], color=color, linewidth=1.5)

ax.set_xlim(2025, 2040)
ax.margins(x=0)
plt.xticks(range(2025, 2041, 3))

y_max_value = max(df[f'Max_Employment_S{s}'].max() for s in range(1, 5))
if STEEL_COLOURS == 'bright':
    # Start from zero with 5 ticks only
    plt.ylim(0, y_max_value)
    plt.yticks(np.linspace(0, y_max_value, 5))
    steel_ylabel = 'Number of Jobs Created'
else:
    # Ticks in multiples of 5,000
    y_max_tick = np.ceil(y_max_value / 5000) * 5000
    plt.ylim(0, y_max_tick)
    plt.yticks(np.arange(0, y_max_tick + 1, 5000))
    steel_ylabel = 'Number of Jobs'
ax.yaxis.set_major_formatter(FuncFormatter(lambda x, pos: f'{int(x):,}'))

ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['bottom'].set_alpha(0.4)
ax.spines['left'].set_alpha(0.4)
plt.xlabel('Year')
plt.ylabel(steel_ylabel)
plt.tight_layout()
plt.gcf().set_size_inches(7.39, 3.7)
os.makedirs(os.path.join(REPORT_DIR, 'plots'), exist_ok=True)  # Separate from Step 6's steel_scenarios.svg
plt.savefig(os.path.join(REPORT_DIR, 'plots', 'steel_scenarios.svg'), format="svg")
plt.show()

# -----------------------------------------------------------------------------
# 4. Building energy efficiency, government plan (BEE.py, stacked_ems.py) - needs ems.xlsx
# -----------------------------------------------------------------------------
if not os.path.exists(EMS_FILE):
    print('Skipped the Building Energy Efficiency figures: ems.xlsx was not uploaded in Step 1.')
else:
    df = pd.read_excel(EMS_FILE, sheet_name='Gov', engine='openpyxl')
    df = df[pd.to_numeric(df['Years'], errors='coerce').notna()]
    df['Years'] = df['Years'].astype(int)
    df[['Max_Gov', 'Med_Gov', 'Min_Gov']] = df[['Max_Gov', 'Med_Gov', 'Min_Gov']].apply(pd.to_numeric, errors='coerce')
    years = df['Years']
    ems_yaxis = REPORT_YAXIS['ems']
    ems_yticks = [ems_yaxis['interval'] * i for i in range(5)]

    def ems_axes(ax, tight_labels):
        ax.set_xlim(years.min(), years.max())
        plt.xticks(range(years.min(), years.max() + 1, 2), fontsize=4)
        plt.ylim(0, ems_yaxis['new_max'])
        plt.yticks(ems_yticks, fontsize=4)
        ax.yaxis.set_major_formatter(FuncFormatter(hundred_formatter))
        for spine in ax.spines.values():
            spine.set_linewidth(1)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.spines['bottom'].set_alpha(0.4)
        ax.spines['left'].set_alpha(0.4)
        labelpad = 1 if tight_labels else None
        plt.xlabel('Year', fontsize=4, labelpad=labelpad)
        plt.ylabel('Number of Jobs', fontsize=4, labelpad=labelpad)

    # BEE.py: min-max band with median line
    color = '#152570'
    plt.figure(figsize=(3.2, 1.5))
    ax = plt.gca()
    ax.fill_between(years, df['Min_Gov'], df['Max_Gov'], color=color, alpha=0.3)
    ax.plot(years, df['Min_Gov'], linestyle='--', color=color, alpha=0.4, linewidth=0.6)
    ax.plot(years, df['Max_Gov'], linestyle='--', color=color, alpha=0.4, linewidth=0.6)
    ax.plot(years, df['Med_Gov'], color=color, linewidth=0.6)
    ems_axes(ax, tight_labels=True)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, 'ems.svg'), format="svg", transparent=True)
    plt.show()

    # stacked_ems.py: min, min-to-median and median-to-max as stacked layers
    plt.figure(figsize=(3.2, 1.5))
    ax = plt.gca()
    ax.stackplot(years, df['Min_Gov'], df['Med_Gov'] - df['Min_Gov'], df['Max_Gov'] - df['Med_Gov'],
                 colors=['#a1b5d9', '#7082b0', '#152570'])
    ems_axes(ax, tight_labels=False)
    plt.savefig(os.path.join(REPORT_DIR, 'ems_area_chart.svg'), format="svg")
    plt.show()

print(f'Report figures saved to {REPORT_DIR}')

## Step 8 - Download the results
Downloads Output_data.zip with raw_data.xlsx, all Excel outputs and the charts.

In [ ]:
# =============================================================================
# STEP 8 - DOWNLOAD ALL RESULTS
# Zips the whole Output_data folder (raw_data.xlsx, all Excel outputs and charts).
# With USE_GOOGLE_DRIVE = True the results are already in your Drive.
# =============================================================================
if IN_COLAB and not USE_GOOGLE_DRIVE:
    zip_path = shutil.make_archive(os.path.join(WORK_DIR, 'Output_data'), 'zip', OUTPUT_DIR)
    files.download(zip_path)
else:
    print(f'Results are in {OUTPUT_DIR}')